# Embedding Engine

## Objective

Convert standardized Chunk Objects into standardized Embedding Objects.

Input

Data/chunks/

↓

chunk_objects.json

Output

Data/embeddings/

document_embeddings.json

This engine does not perform:

- Document ingestion
- Text extraction
- Chunking
- Retrieval
- Generation

Its only responsibility is generating embeddings.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path

from sentence_transformers import SentenceTransformer

print("Environment Ready")

Environment Ready


In [ ]:
ROOT = Path("/content/drive/MyDrive/MicroBrain")

CHUNKS = ROOT / "Data" / "chunks"

EMBEDDINGS = ROOT / "Data" / "embeddings"

METADATA = ROOT / "Data" / "metadata"

In [ ]:
print(CHUNKS)

print(EMBEDDINGS)

/content/drive/MyDrive/MicroBrain/Data/chunks
/content/drive/MyDrive/MicroBrain/Data/embeddings


In [ ]:
chunk_files = sorted(

    CHUNKS.glob("*_chunks.json")

)

print(len(chunk_files))

1


In [ ]:
chunk_file = chunk_files[0]

with open(chunk_file, "r", encoding="utf-8") as file:

    chunk_objects = json.load(file)

In [ ]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

In [ ]:
embedding_model = SentenceTransformer(

    MODEL_NAME

)

print("Embedding Model Ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Ready


In [ ]:
print(

chunk_objects[0]["content"]["text"][:250]

)

From Words to Answers How a language model turns your question into a response, and how Retrieval-Augmented Generation (RAG) gives it new knowledge without retraining it — explained simply, with the underlying math alongside. The Big Picture There ar


# Step 7 - Standard Embedding Object

Every embedding is represented as a structured object.

Future retrieval engines consume this object instead of raw vectors.

In [ ]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

EMBEDDING_DIMENSION = 384

In [15]:
embedding_objects = []

In [16]:
for chunk in chunk_objects:

    vector = embedding_model.encode(

        chunk["content"]["text"]

    )

    embedding = {

        "chunk_id": chunk["chunk_id"],

        "document_id": chunk["document_id"],

        "chunk_index": chunk["chunk_index"],

        "model": MODEL_NAME,

        "dimension": EMBEDDING_DIMENSION,

        "vector": vector.tolist()

    }

    embedding_objects.append(embedding)

In [17]:
print(type(embedding_objects))

print()

print(len(embedding_objects))

<class 'list'>

25


# Step 8 - Create Embedding Objects

Instead of saving a raw NumPy array, create a standardized embedding object for every chunk.

Each object links together:

- Document ID
- Chunk ID
- Embedding Vector
- Embedding Model
- Vector Dimension

Future retrieval engines will load these objects directly without recomputing embeddings.

Output

embedding_objects

In [21]:
print(chunk_objects[0].keys())


dict_keys(['chunk_id', 'document_id', 'chunk_index', 'start_character', 'end_character', 'metadata', 'content'])


In [22]:
print(chunk_objects[0]["content"].keys())

dict_keys(['text'])


In [23]:
chunk_texts = []

for chunk in chunk_objects:

    chunk_texts.append(
        chunk["content"]["text"]
    )

print(type(chunk_texts))
print(len(chunk_texts))


<class 'list'>
25


In [24]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
print(type(chunk_embeddings))
print()

print(chunk_embeddings.shape)

<class 'numpy.ndarray'>

(25, 384)


In [27]:
embedding_objects = []

for chunk, embedding in zip(chunk_objects, chunk_embeddings):

    embedding_objects.append({

        "embedding_id": chunk["chunk_id"],

        "document_id": chunk["document_id"],

        "chunk_id": chunk["chunk_id"],

        "model": MODEL_NAME,

        "dimensions": int(embedding.shape[0]),

        "vector": embedding.tolist()

    })

print(type(embedding_objects))
print()
print(len(embedding_objects))

<class 'list'>

25


# Step 9 - Validate Embedding Object

Verify the structure before saving.

In [28]:
print(embedding_objects[0].keys())

dict_keys(['embedding_id', 'document_id', 'chunk_id', 'model', 'dimensions', 'vector'])


In [29]:
print(embedding_objects[0]["embedding_id"])

print()

print(embedding_objects[0]["document_id"])

print()

print(embedding_objects[0]["model"])

print()

print(embedding_objects[0]["dimensions"])

print()

print(len(embedding_objects[0]["vector"]))

bb15390c-fcd7-4472-8796-db06731d0843

a60be88b-ec91-429e-8ab4-c1bb43cdf6e6

BAAI/bge-small-en-v1.5

384

384


# Step 10 - Save Embedding Objects

Persist the embedding objects so future engines can load them directly.

Output

Data/embeddings/<document_id>_embeddings.json

In [32]:
import json

# Every chunk belongs to the same document
document_id = chunk_objects[0]["document_id"]

embedding_path = EMBEDDINGS / f"{document_id}_embeddings.json"

with open(embedding_path, "w", encoding="utf-8") as file:

    json.dump(
        embedding_objects,
        file,
        indent=4
    )

print("Saved")

print()

print(embedding_path)

Saved

/content/drive/MyDrive/MicroBrain/Data/embeddings/a60be88b-ec91-429e-8ab4-c1bb43cdf6e6_embeddings.json


# Step 11 - Update Embedding Metadata

Maintain a lightweight registry of all embedding files.

Future retrieval engines will use this registry to discover available embedding datasets.

In [33]:
embedding_registry = {

    "documents": [

        {

            "document_id": document_id,

            "embedding_file": embedding_path.name,

            "embedding_model": MODEL_NAME,

            "embedding_dimension": EMBEDDING_DIMENSION,

            "total_embeddings": len(embedding_objects)

        }

    ]

}

metadata_path = METADATA / "embeddings.json"

with open(metadata_path, "w", encoding="utf-8") as file:

    json.dump(
        embedding_registry,
        file,
        indent=4
    )

print("Metadata Updated")

print()

print(metadata_path)

Metadata Updated

/content/drive/MyDrive/MicroBrain/Data/metadata/embeddings.json


# Step 12 - Final Validation

Verify that the Embedding Engine produced all expected outputs.

In [34]:
print("Embedding Files")
print()

for file in EMBEDDINGS.glob("*.json"):
    print(file.name)

print()

print("Metadata Files")
print()

for file in METADATA.glob("*embeddings*.json"):
    print(file.name)

Embedding Files

a60be88b-ec91-429e-8ab4-c1bb43cdf6e6_embeddings.json

Metadata Files

embeddings.json
